# Generate Initial Catalog Candidates

Propagates the historical catalog to each detected fragment and saves the candidate tables required for event consolidation. Final association labels are assigned only after consolidation.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


## 1. Set repository paths


## 2. Install only the catalog-propagation dependencies


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "sgp4": "sgp4>=2.23",
    "astropy": "astropy>=6.1",
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *missing
    ])
else:
    print("sgp4 and astropy are already available.")


## 3. Freeze the input and load the 24 hourly object-detection files


In [ ]:
from pathlib import Path
import hashlib
import gzip
import json
import os

import numpy as np
import pandas as pd

BASE = Path(os.environ.get("DEBRIS_BASE", str(PROJECT_ROOT)))
BENCHMARK_ROOT = BASE / "phase0_outputs/detector_benchmark_raw_swerling_v2"
HOURLY_ROOT = BENCHMARK_ROOT / "hourly"
SELECTED_DETECTOR_PATH = BENCHMARK_ROOT / "SELECTED_DETECTOR.json"
HISTORY_PATH = (
    BASE / "phase0_outputs/catalog_matching/historical_catalog"
    / "gp_history_20180103_20180106.json.gz"
)
OUT = (
    BASE / "phase0_outputs/catalog_matching/ca_cfar_1p75_24h"
)
CHECKPOINTS = OUT / "checkpoints"
OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

EXPECTED_CONFIG_ID = "ca_1.75"
EXPECTED_METHOD = "CA-CFAR"
EXPECTED_THRESHOLD = 1.75
EXPECTED_TRACK_RULE = "short_0p8"

PAPER_RANGE_FLOOR_KM = 1.0
PAPER_RANGE_RATE_FLOOR_KM_S = 0.020
BEAM_MODEL_LIMIT_DEG = 5.0
PRIMARY_RANGE_FRACTION = 0.001                                     
PRIMARY_RANGE_RATE_FRACTION = 0.01                              
EQUAL_ONE_PERCENT_FRACTION = 0.01                                     
RATE_PERCENT_FLOOR_KM_S = 2.0                                               
TOP_CANDIDATES_TO_SAVE = 25

GMF_MIN_RANGE_GATE = 6800
GMF_MAX_RANGE_GATE = 7280
GMF_RANGE_GATE_SOURCE = (
    "Inherited from the Hardtarget example configuration; not campaign-derived"
)

BEAM_AZIMUTH_DEG = 90.0
BEAM_ELEVATION_DEG = 75.0

                                                                      
GLOBAL_TIME_OFFSET_SEC = 0.0
FORCE_RECOMPUTE = False

required_paths = [HOURLY_ROOT, SELECTED_DETECTOR_PATH, HISTORY_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required input paths:\n" + "\n".join(missing_paths))

selected_payload = json.loads(SELECTED_DETECTOR_PATH.read_text())
frozen = selected_payload.get("selection", selected_payload)
frozen_config = str(frozen.get("config_id", frozen.get("chosen_config_id", "")))
frozen_rule = str(frozen.get("track_rule", frozen.get("chosen_track_rule", "")))
frozen_threshold = float(frozen.get("threshold", frozen.get("chosen_threshold", np.nan)))
if frozen_config != EXPECTED_CONFIG_ID:
    raise RuntimeError(
        f"Frozen detector is {frozen_config!r}, not {EXPECTED_CONFIG_ID!r}. "
        "Do not silently catalog a different detector."
    )
if frozen_rule != EXPECTED_TRACK_RULE or not np.isclose(
    frozen_threshold, EXPECTED_THRESHOLD
):
    raise RuntimeError(
        f"Frozen selection is {frozen_config}/{frozen_rule}/{frozen_threshold}, "
        f"expected {EXPECTED_CONFIG_ID}/{EXPECTED_TRACK_RULE}/{EXPECTED_THRESHOLD}."
    )

hourly_files = sorted(HOURLY_ROOT.glob("*/real_tracks.csv.gz"))
if len(hourly_files) != 24:
    raise RuntimeError(
        f"Expected exactly 24 hourly real_tracks.csv.gz files, found {len(hourly_files)}."
    )

frames = []
for path in hourly_files:
    frame = pd.read_csv(path)
    frame["source_hour_folder"] = path.parent.name
    frames.append(frame)
all_detector_rows = pd.concat(frames, ignore_index=True)

required_columns = [
    "hour_key", "config_id", "method", "threshold", "track_rule",
    "start_time_unix", "end_time_unix", "duration_sec",
    "number_unique_times", "minimum_range_km", "maximum_range_km",
    "reference_range_km", "velocity_center_km_s", "acceleration_km_s2",
    "trajectory_rmse_km", "peak_detector_score",
]
absent = [column for column in required_columns if column not in all_detector_rows]
if absent:
    raise KeyError("Hourly object-detection files are missing: " + ", ".join(absent))

objects = all_detector_rows[
    all_detector_rows["config_id"].astype(str).eq(EXPECTED_CONFIG_ID)
    & all_detector_rows["track_rule"].astype(str).eq(EXPECTED_TRACK_RULE)
    & np.isclose(pd.to_numeric(all_detector_rows["threshold"]), EXPECTED_THRESHOLD)
].copy()
if objects.empty:
    raise RuntimeError("No rows match the frozen CA-CFAR operating point.")

objects = objects.sort_values(
    ["start_time_unix", "end_time_unix", "reference_range_km"]
).reset_index(drop=True)
objects.insert(0, "object_detection_id", [f"D{i:04d}" for i in range(1, len(objects) + 1)])
objects["reference_time_unix"] = 0.5 * (
    pd.to_numeric(objects["start_time_unix"])
    + pd.to_numeric(objects["end_time_unix"])
)
objects["reference_time_unix"] += GLOBAL_TIME_OFFSET_SEC
objects["observed_range_km"] = pd.to_numeric(objects["reference_range_km"])
objects["observed_range_rate_km_s"] = pd.to_numeric(objects["velocity_center_km_s"])
objects["beam_azimuth_deg"] = BEAM_AZIMUTH_DEG
objects["beam_elevation_deg"] = BEAM_ELEVATION_DEG
objects["reference_time_utc"] = pd.to_datetime(
    objects["reference_time_unix"], unit="s", utc=True
).dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")

inventory_path = OUT / "selected_object_detections.csv"
objects.to_csv(inventory_path, index=False)

print("Frozen detector:", EXPECTED_CONFIG_ID, EXPECTED_TRACK_RULE)
print("Completed hourly files:", len(hourly_files))
print("Selected object detections:", len(objects))
print(
    "Observed CFAR-valid range represented by detections: "
    f"{objects.minimum_range_km.min():.3f} to {objects.maximum_range_km.max():.3f} km"
)
print("Inventory saved:", inventory_path)
display(objects[[
    "object_detection_id", "hour_key", "reference_time_utc",
    "observed_range_km", "observed_range_rate_km_s", "duration_sec",
    "peak_detector_score"
]].head(10))


## 4. Exact-time historical catalog propagation helpers


In [ ]:
import warnings

import astropy.units as u
from astropy.coordinates import (
    CartesianDifferential,
    CartesianRepresentation,
    EarthLocation,
    ITRS,
    TEME,
)
from astropy.time import Time
from astropy.utils import iers
from sgp4.api import Satrec, SatrecArray

iers.conf.auto_download = False
iers.conf.auto_max_age = None

RADAR_LAT_DEG = 69.0 + 35.0 / 60.0 + 11.0 / 3600.0
RADAR_LON_DEG = 19.0 + 13.0 / 60.0 + 38.0 / 3600.0
RADAR_ALT_M = 86.0

station = EarthLocation.from_geodetic(
    lon=RADAR_LON_DEG * u.deg,
    lat=RADAR_LAT_DEG * u.deg,
    height=RADAR_ALT_M * u.m,
)

with gzip.open(HISTORY_PATH, "rt", encoding="utf-8") as handle:
    history_records = json.load(handle)
history = pd.DataFrame(history_records)
required_history = ["NORAD_CAT_ID", "EPOCH", "TLE_LINE1", "TLE_LINE2"]
missing_history = [column for column in required_history if column not in history]
if missing_history:
    raise KeyError("Historical cache is missing: " + ", ".join(missing_history))

history["NORAD_CAT_ID"] = pd.to_numeric(
    history["NORAD_CAT_ID"], errors="coerce"
).astype("Int64")
history["EPOCH_UTC"] = pd.to_datetime(history["EPOCH"], utc=True, errors="coerce")
history = history.dropna(
    subset=["NORAD_CAT_ID", "EPOCH_UTC", "TLE_LINE1", "TLE_LINE2"]
).sort_values(["NORAD_CAT_ID", "EPOCH_UTC"]).reset_index(drop=True)

                                                                      
SATREC_CACHE = {}


def teme_to_radar_geometry(r_km, v_km_s, obstime, beam_az_deg, beam_el_deg):
    r_km = np.asarray(r_km, dtype=float)
    v_km_s = np.asarray(v_km_s, dtype=float)
    if r_km.ndim == 1:
        r_km = r_km[None, :]
        v_km_s = v_km_s[None, :]

    representation = CartesianRepresentation(
        x=r_km[:, 0] * u.km,
        y=r_km[:, 1] * u.km,
        z=r_km[:, 2] * u.km,
        differentials=CartesianDifferential(
            d_x=v_km_s[:, 0] * u.km / u.s,
            d_y=v_km_s[:, 1] * u.km / u.s,
            d_z=v_km_s[:, 2] * u.km / u.s,
        ),
    )
    teme = TEME(representation, obstime=obstime)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        itrs = teme.transform_to(ITRS(obstime=obstime))

    satellite_position = itrs.cartesian.xyz.to_value(u.km).T
    satellite_velocity = itrs.cartesian.differentials["s"].d_xyz.to_value(
        u.km / u.s
    ).T
    station_position = station.get_itrs(obstime=obstime).cartesian.xyz.to_value(u.km).T
    if station_position.ndim == 1:
        station_position = station_position[None, :]

    relative_position = satellite_position - station_position
    slant_range_km = np.linalg.norm(relative_position, axis=1)
    radial_rate_km_s = np.einsum(
        "ij,ij->i", relative_position, satellite_velocity
    ) / slant_range_km

    lat = np.deg2rad(RADAR_LAT_DEG)
    lon = np.deg2rad(RADAR_LON_DEG)
    x, y, z = relative_position.T
    east = -np.sin(lon) * x + np.cos(lon) * y
    north = (
        -np.sin(lat) * np.cos(lon) * x
        - np.sin(lat) * np.sin(lon) * y
        + np.cos(lat) * z
    )
    up = (
        np.cos(lat) * np.cos(lon) * x
        + np.cos(lat) * np.sin(lon) * y
        + np.sin(lat) * z
    )
    azimuth_deg = np.rad2deg(np.arctan2(east, north)) % 360.0
    elevation_deg = np.rad2deg(
        np.arcsin(np.clip(up / slant_range_km, -1.0, 1.0))
    )

    az = np.deg2rad(azimuth_deg)
    el = np.deg2rad(elevation_deg)
    beam_az = np.deg2rad(float(beam_az_deg))
    beam_el = np.deg2rad(float(beam_el_deg))
    los_e = np.cos(el) * np.sin(az)
    los_n = np.cos(el) * np.cos(az)
    los_u = np.sin(el)
    beam_e = np.cos(beam_el) * np.sin(beam_az)
    beam_n = np.cos(beam_el) * np.cos(beam_az)
    beam_u = np.sin(beam_el)
    cosine = np.clip(
        los_e * beam_e + los_n * beam_n + los_u * beam_u, -1.0, 1.0
    )
    beam_separation_deg = np.rad2deg(np.arccos(cosine))
    return {
        "slant_range_km": slant_range_km,
        "radial_rate_km_s": radial_rate_km_s,
        "azimuth_deg": azimuth_deg,
        "elevation_deg": elevation_deg,
        "beam_separation_deg": beam_separation_deg,
    }


def latest_prior_catalog(event_unix):
    event_timestamp = pd.to_datetime(float(event_unix), unit="s", utc=True)
    prior = history[history["EPOCH_UTC"] <= event_timestamp]
    if prior.empty:
        raise RuntimeError("The history cache contains no TLE prior to this event.")
                                                                              
    selected = prior.drop_duplicates("NORAD_CAT_ID", keep="last").copy()
    selected["tle_age_hours"] = (
        event_timestamp - selected["EPOCH_UTC"]
    ).dt.total_seconds() / 3600.0
    return selected.reset_index(drop=True)


def build_satellite_array(catalog):
    satrecs = []
    kept_rows = []
    failures = 0
    for row_index, row in catalog.iterrows():
        key = (str(row["TLE_LINE1"]), str(row["TLE_LINE2"]))
        try:
            sat = SATREC_CACHE.get(key)
            if sat is None:
                sat = Satrec.twoline2rv(*key)
                SATREC_CACHE[key] = sat
            satrecs.append(sat)
            kept_rows.append(row_index)
        except Exception:
            failures += 1
    if not satrecs:
        raise RuntimeError("No latest-prior TLEs could be parsed.")
    return catalog.loc[kept_rows].reset_index(drop=True), satrecs, failures


def propagate_all(satrecs, obstime, beam_az, beam_el):
    sat_array = SatrecArray(satrecs)
    error, r, v = sat_array.sgp4(
        np.asarray([obstime.jd1], dtype=float),
        np.asarray([obstime.jd2], dtype=float),
    )
    error = np.asarray(error)[:, 0]
    r = np.asarray(r)[:, 0, :]
    v = np.asarray(v)[:, 0, :]
    valid = (
        (error == 0)
        & np.all(np.isfinite(r), axis=1)
        & np.all(np.isfinite(v), axis=1)
    )
    geometry = teme_to_radar_geometry(
        r[valid], v[valid], obstime, beam_az, beam_el
    )
    return np.where(valid)[0], geometry


def criterion_status(number):
    if int(number) == 1:
        return "unique"
    if int(number) > 1:
        return "ambiguous"
    return "none"


CONFIG_PAYLOAD = {
    "version": "ca_cfar_1p75_catalog_v2_hybrid_relative",
    "config_id": EXPECTED_CONFIG_ID,
    "track_rule": EXPECTED_TRACK_RULE,
    "paper_range_floor_km": PAPER_RANGE_FLOOR_KM,
    "paper_range_rate_floor_km_s": PAPER_RANGE_RATE_FLOOR_KM_S,
    "beam_limit_deg": BEAM_MODEL_LIMIT_DEG,
    "primary_range_fraction": PRIMARY_RANGE_FRACTION,
    "primary_range_rate_fraction": PRIMARY_RANGE_RATE_FRACTION,
    "equal_one_percent_fraction": EQUAL_ONE_PERCENT_FRACTION,
    "rate_percent_floor_km_s": RATE_PERCENT_FLOOR_KM_S,
    "time_offset_sec": GLOBAL_TIME_OFFSET_SEC,
}
CONFIG_HASH = hashlib.sha256(
    json.dumps(CONFIG_PAYLOAD, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

print(f"Historical records: {len(history):,}")
print(f"Historical catalog objects: {history.NORAD_CAT_ID.nunique():,}")
print("Checkpoint configuration:", CONFIG_HASH)


## 5. Run or resume all 24-hour associations


In [ ]:
import time


def row_signature(row):
    payload = {
        key: float(row[key]) if key != "object_detection_id" else str(row[key])
        for key in [
            "object_detection_id", "reference_time_unix", "observed_range_km",
            "observed_range_rate_km_s", "beam_azimuth_deg", "beam_elevation_deg"
        ]
    }
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode("utf-8")
    ).hexdigest()[:16]


def atomic_json(payload, path):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str))
    temp.replace(path)


def atomic_gzip_csv(frame, path):
    temp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp, index=False, compression="gzip")
    temp.replace(path)


def process_one_detection(row):
    detection_id = str(row["object_detection_id"])
    center_unix = float(row["reference_time_unix"])
    obstime = Time(center_unix, format="unix", scale="utc")
    observed_range = float(row["observed_range_km"])
    observed_rate = float(row["observed_range_rate_km_s"])
    beam_az = float(row["beam_azimuth_deg"])
    beam_el = float(row["beam_elevation_deg"])

    prior_catalog = latest_prior_catalog(center_unix)
    prior_catalog, satrecs, parse_failures = build_satellite_array(prior_catalog)
    valid_indices, geometry = propagate_all(satrecs, obstime, beam_az, beam_el)
    propagated = prior_catalog.iloc[valid_indices].copy().reset_index(drop=True)

    propagated["predicted_slant_range_km"] = geometry["slant_range_km"]
    propagated["predicted_radial_rate_km_s"] = geometry["radial_rate_km_s"]
    propagated["predicted_azimuth_deg"] = geometry["azimuth_deg"]
    propagated["predicted_elevation_deg"] = geometry["elevation_deg"]
    propagated["beam_separation_deg"] = geometry["beam_separation_deg"]
    propagated["range_residual_km"] = (
        propagated["predicted_slant_range_km"] - observed_range
    )
    propagated["range_rate_residual_km_s"] = (
        propagated["predicted_radial_rate_km_s"] - observed_rate
    )
    propagated["absolute_range_fraction"] = (
        propagated["range_residual_km"].abs() / observed_range
    )
    propagated["range_residual_percent"] = (
        100.0 * propagated["absolute_range_fraction"]
    )
    raw_rate_denominator = abs(observed_rate)
    if raw_rate_denominator > 0.0:
        propagated["absolute_range_rate_fraction_raw"] = (
            propagated["range_rate_residual_km_s"].abs()
            / raw_rate_denominator
        )
    else:
        propagated["absolute_range_rate_fraction_raw"] = np.inf
    propagated["range_rate_residual_percent_raw"] = (
        100.0 * propagated["absolute_range_rate_fraction_raw"]
    )

    primary_range_gate_km = max(
        PAPER_RANGE_FLOOR_KM,
        PRIMARY_RANGE_FRACTION * observed_range,
    )
    primary_range_rate_gate_km_s = max(
        PAPER_RANGE_RATE_FLOOR_KM_S,
        PRIMARY_RANGE_RATE_FRACTION * abs(observed_rate),
    )
    equal_one_percent_range_gate_km = (
        EQUAL_ONE_PERCENT_FRACTION * observed_range
    )
    equal_one_percent_rate_gate_km_s = (
        EQUAL_ONE_PERCENT_FRACTION
        * max(abs(observed_rate), RATE_PERCENT_FLOOR_KM_S)
    )
    propagated["primary_range_gate_km"] = primary_range_gate_km
    propagated["primary_range_rate_gate_km_s"] = primary_range_rate_gate_km_s
    propagated["equal_one_percent_range_gate_km"] = (
        equal_one_percent_range_gate_km
    )
    propagated["equal_one_percent_rate_gate_km_s"] = (
        equal_one_percent_rate_gate_km_s
    )
    propagated["within_beam_model"] = (
        propagated["beam_separation_deg"] <= BEAM_MODEL_LIMIT_DEG
    )
    propagated["above_horizon"] = propagated["predicted_elevation_deg"] > 0.0
    common = (
        propagated["within_beam_model"]
        & propagated["above_horizon"]
    )
    propagated["meets_paper_fixed_gates"] = (
        (propagated["range_residual_km"].abs() < PAPER_RANGE_FLOOR_KM)
        & (
            propagated["range_rate_residual_km_s"].abs()
            < PAPER_RANGE_RATE_FLOOR_KM_S
        )
        & common
    )
    propagated["meets_primary_hybrid_gates"] = (
        (propagated["range_residual_km"].abs() < primary_range_gate_km)
        & (
            propagated["range_rate_residual_km_s"].abs()
            < primary_range_rate_gate_km_s
        )
        & common
    )
    propagated["meets_equal_one_percent_diagnostic"] = (
        (
            propagated["range_residual_km"].abs()
            < equal_one_percent_range_gate_km
        )
        & (
            propagated["range_rate_residual_km_s"].abs()
            < equal_one_percent_rate_gate_km_s
        )
        & common
    )
    propagated["association_metric_primary"] = np.sqrt(
        (propagated["range_residual_km"] / primary_range_gate_km) ** 2
        + (
            propagated["range_rate_residual_km_s"]
            / primary_range_rate_gate_km_s
        ) ** 2
    )
    propagated["association_metric_paper"] = np.sqrt(
        (propagated["range_residual_km"] / PAPER_RANGE_FLOOR_KM) ** 2
        + (
            propagated["range_rate_residual_km_s"]
            / PAPER_RANGE_RATE_FLOOR_KM_S
        ) ** 2
    )
    propagated = propagated.sort_values(
        "association_metric_primary"
    ).reset_index(drop=True)
    propagated["rank_within_detection"] = np.arange(1, len(propagated) + 1)

    n_primary = int(propagated["meets_primary_hybrid_gates"].sum())
    n_fixed = int(propagated["meets_paper_fixed_gates"].sum())
    n_equal_one = int(
        propagated["meets_equal_one_percent_diagnostic"].sum()
    )
    if n_primary:
        best = propagated[propagated["meets_primary_hybrid_gates"]].iloc[0]
    elif n_fixed:
        best = propagated[propagated["meets_paper_fixed_gates"]].iloc[0]
    else:
        best = propagated.iloc[0]

    primary_status = criterion_status(n_primary)
    fixed_status = criterion_status(n_fixed)
    equal_one_status = criterion_status(n_equal_one)
    is_unique_primary = primary_status == "unique"
    paper_unique = (
        propagated[propagated["meets_paper_fixed_gates"]].iloc[0]
        if n_fixed == 1 else None
    )
    equal_one_unique = (
        propagated[propagated["meets_equal_one_percent_diagnostic"]].iloc[0]
        if n_equal_one == 1 else None
    )
    summary = {
        "config_hash": CONFIG_HASH,
        "row_signature": row_signature(row),
        "object_detection_id": detection_id,
        "hour_key": str(row["hour_key"]),
        "reference_time_unix": center_unix,
        "reference_time_utc": obstime.utc.isot,
        "duration_sec": float(row["duration_sec"]),
        "number_unique_times": int(row["number_unique_times"]),
        "peak_detector_score": float(row["peak_detector_score"]),
        "observed_range_km": observed_range,
        "observed_range_rate_km_s": observed_rate,
        "observed_acceleration_km_s2": float(row["acceleration_km_s2"]),
        "trajectory_rmse_km": float(row["trajectory_rmse_km"]),
        "primary_hybrid_status": primary_status,
        "paper_fixed_status": fixed_status,
        "equal_one_percent_diagnostic_status": equal_one_status,
        "number_meeting_primary_hybrid": n_primary,
        "number_meeting_fixed_1km": n_fixed,
        "number_meeting_equal_one_percent_diagnostic": n_equal_one,
        "primary_range_gate_km": primary_range_gate_km,
        "primary_range_rate_gate_km_s": primary_range_rate_gate_km_s,
        "equal_one_percent_range_gate_km": equal_one_percent_range_gate_km,
        "equal_one_percent_rate_gate_km_s": equal_one_percent_rate_gate_km_s,
        "catalog_objects_with_prior_tle": int(len(prior_catalog)),
        "catalog_objects_propagated": int(len(propagated)),
        "tle_parse_failures": int(parse_failures),
                                                                              
        "matched_NORAD_CAT_ID": int(best["NORAD_CAT_ID"]) if is_unique_primary else None,
        "matched_OBJECT_NAME": str(best.get("OBJECT_NAME", "")) if is_unique_primary else None,
        "matched_OBJECT_ID": str(best.get("OBJECT_ID", "")) if is_unique_primary else None,
        "matched_OBJECT_TYPE": str(best.get("OBJECT_TYPE", "")) if is_unique_primary else None,
        "matched_RCS_SIZE": str(best.get("RCS_SIZE", "")) if is_unique_primary else None,
        "paper_unique_NORAD_CAT_ID": (
            int(paper_unique["NORAD_CAT_ID"]) if paper_unique is not None else None
        ),
        "equal_one_percent_unique_NORAD_CAT_ID": (
            int(equal_one_unique["NORAD_CAT_ID"])
            if equal_one_unique is not None else None
        ),
                                                                                 
        "diagnostic_best_NORAD_CAT_ID": int(best["NORAD_CAT_ID"]),
        "diagnostic_best_OBJECT_NAME": str(best.get("OBJECT_NAME", "")),
        "TLE_EPOCH_UTC": str(best.get("EPOCH_UTC", "")),
        "tle_age_hours": float(best.get("tle_age_hours", np.nan)),
        "predicted_slant_range_km": float(best["predicted_slant_range_km"]),
        "range_residual_km": float(best["range_residual_km"]),
        "range_residual_percent": float(best["range_residual_percent"]),
        "predicted_radial_rate_km_s": float(best["predicted_radial_rate_km_s"]),
        "range_rate_residual_km_s": float(best["range_rate_residual_km_s"]),
        "range_rate_residual_percent_raw": float(
            best["range_rate_residual_percent_raw"]
        ),
        "normalized_primary_range_residual": float(
            best["range_residual_km"] / primary_range_gate_km
        ),
        "normalized_primary_range_rate_residual": float(
            best["range_rate_residual_km_s"]
            / primary_range_rate_gate_km_s
        ),
        "predicted_azimuth_deg": float(best["predicted_azimuth_deg"]),
        "predicted_elevation_deg": float(best["predicted_elevation_deg"]),
        "beam_separation_deg": float(best["beam_separation_deg"]),
        "best_candidate_metric_primary": float(
            best["association_metric_primary"]
        ),
        "best_candidate_metric_paper": float(best["association_metric_paper"]),
    }

    save_rows = propagated[
        (propagated["rank_within_detection"] <= TOP_CANDIDATES_TO_SAVE)
        | propagated["meets_primary_hybrid_gates"]
        | propagated["meets_paper_fixed_gates"]
        | propagated["meets_equal_one_percent_diagnostic"]
    ].copy()
    save_rows.insert(0, "object_detection_id", detection_id)
    save_rows.insert(1, "reference_time_utc", obstime.utc.isot)
    save_rows.insert(2, "observed_range_km", observed_range)
    save_rows.insert(3, "observed_range_rate_km_s", observed_rate)
    return summary, save_rows


summary_rows = []
candidate_frames = []
start_clock = time.perf_counter()
resumed = 0
compute_durations = []

for index, (_, row) in enumerate(objects.iterrows(), start=1):
    detection_id = str(row["object_detection_id"])
    summary_path = CHECKPOINTS / f"{detection_id}_summary.json"
    candidates_path = CHECKPOINTS / f"{detection_id}_candidates.csv.gz"
    reusable = False
    if not FORCE_RECOMPUTE and summary_path.exists() and candidates_path.exists():
        try:
            cached = json.loads(summary_path.read_text())
            reusable = (
                cached.get("config_hash") == CONFIG_HASH
                and cached.get("row_signature") == row_signature(row)
            )
        except Exception:
            reusable = False

    if reusable:
        summary = cached
        candidates = pd.read_csv(candidates_path)
        resumed += 1
        mode = "checkpoint"
    else:
        event_clock = time.perf_counter()
        summary, candidates = process_one_detection(row)
        atomic_gzip_csv(candidates, candidates_path)
        atomic_json(summary, summary_path)
        event_duration = time.perf_counter() - event_clock
        compute_durations.append(event_duration)
        mode = f"computed {event_duration:.1f}s"

    summary_rows.append(summary)
    candidate_frames.append(candidates)
    elapsed = time.perf_counter() - start_clock
    if compute_durations:
        eta_minutes = (
            float(np.median(compute_durations[-10:]))
            * (len(objects) - index) / 60.0
        )
        eta_text = f"; rough ETA={eta_minutes:.1f} min"
    else:
        eta_text = ""
    print(
        f"{detection_id} [{index:03d}/{len(objects):03d}] {mode}; "
        f"primary-hybrid={summary['primary_hybrid_status']}, "
        f"paper-fixed={summary['paper_fixed_status']}, "
        f"equal-1%-diagnostic={summary['equal_one_percent_diagnostic_status']}; "
        f"elapsed={elapsed/60:.1f} min{eta_text}"
    )

association_summary = pd.DataFrame(summary_rows).sort_values(
    "reference_time_unix"
).reset_index(drop=True)
catalog_candidates = pd.concat(candidate_frames, ignore_index=True)

SUMMARY_PATH = OUT / "catalog_association_summary.csv"
CANDIDATES_PATH = OUT / "catalog_candidates_top25.csv.gz"
association_summary.to_csv(SUMMARY_PATH, index=False)
catalog_candidates.to_csv(CANDIDATES_PATH, index=False, compression="gzip")

print("\nFinished all detections.")
print("Loaded from checkpoints:", resumed)
print("Total elapsed minutes:", round((time.perf_counter() - start_clock) / 60.0, 2))
print("Summary:", SUMMARY_PATH)
print("Candidates:", CANDIDATES_PATH)


## 6. Campaign tables and 2×2 diagnostic panel


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 300, 'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 9, 'axes.spines.top': False, 'axes.spines.right': False})
criteria = [('Primary hybrid\n0.1% R, 1% rate', 'primary_hybrid_status'), ('Published fixed\n1 km, 20 m/s', 'paper_fixed_status'), ('Requested diagnostic\nequal 1%', 'equal_one_percent_diagnostic_status')]
sensitivity_rows = []
for label, column in criteria:
    counts = association_summary[column].value_counts()
    sensitivity_rows.append({'criterion': label.replace('\n', ' '), 'unique': int(counts.get('unique', 0)), 'ambiguous': int(counts.get('ambiguous', 0)), 'none': int(counts.get('none', 0))})
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUT / 'matching_gate_sensitivity.csv', index=False)
matched = association_summary[association_summary.primary_hybrid_status.eq('unique')].copy()
ambiguous = association_summary[association_summary.primary_hybrid_status.eq('ambiguous')].copy()
unresolved = association_summary[association_summary.primary_hybrid_status.eq('none')].copy()
paper_matched = association_summary[association_summary.paper_fixed_status.eq('unique')].copy()
paper_ambiguous = association_summary[association_summary.paper_fixed_status.eq('ambiguous')].copy()
paper_unresolved = association_summary[association_summary.paper_fixed_status.eq('none')].copy()
unique_catalog = matched.dropna(subset=['matched_NORAD_CAT_ID']).sort_values(['matched_NORAD_CAT_ID', 'reference_time_unix']).drop_duplicates('matched_NORAD_CAT_ID')
unique_catalog.to_csv(OUT / 'unique_matched_catalog_objects.csv', index=False)
by_hour = association_summary.groupby(['hour_key', 'primary_hybrid_status']).size().unstack(fill_value=0).reindex(columns=['unique', 'ambiguous', 'none'], fill_value=0).sort_index()
colors = {'unique': '#176B87', 'ambiguous': '#F2A541', 'none': '#B7C4CF'}
fig, axes = plt.subplots(2, 2, figsize=(13.2, 8.8), constrained_layout=True)
ax = axes[0, 0]
bottom = np.zeros(len(by_hour))
x = np.arange(len(by_hour))
for status in ['unique', 'ambiguous', 'none']:
    values = by_hour[status].to_numpy()
    ax.bar(x, values, bottom=bottom, color=colors[status], label=status.capitalize())
    bottom += values
ax.set_xticks(x[::2])
ax.set_xticklabels([str(value)[-2:] for value in by_hour.index[::2]], rotation=0)
ax.set_xlabel('UTC hour (Jan 4 12 through Jan 5 11)')
ax.set_ylabel('Object detections')
ax.set_title('(a) Primary hybrid catalog status by hour', loc='left', fontweight='bold')
ax.legend(frameon=False, ncol=3)
ax = axes[0, 1]
time_values = pd.to_datetime(association_summary.reference_time_unix, unit='s', utc=True)
for status in ['none', 'ambiguous', 'unique']:
    mask = association_summary.primary_hybrid_status.eq(status)
    ax.scatter(time_values[mask], association_summary.loc[mask, 'observed_range_km'], s=26 if status == 'unique' else 18, alpha=0.82, color=colors[status], label=status.capitalize(), edgecolor='none')
ax.set_ylabel('Observed slant range (km)')
ax.set_xlabel('UTC time')
ax.tick_params(axis='x', rotation=25)
ax.set_title('(b) Frozen CA-CFAR object detections', loc='left', fontweight='bold')
ax = axes[1, 0]
for status in ['none', 'ambiguous', 'unique']:
    mask = association_summary.primary_hybrid_status.eq(status)
    ax.scatter(association_summary.loc[mask, 'normalized_primary_range_residual'], association_summary.loc[mask, 'normalized_primary_range_rate_residual'], s=28 if status == 'unique' else 18, alpha=0.78, color=colors[status], label=status.capitalize(), edgecolor='none')
ax.add_patch(Rectangle((-1.0, -1.0), 2.0, 2.0, fill=False, linewidth=1.7, edgecolor='#8B1E3F', linestyle='--'))
ax.axhline(0, color='0.8', linewidth=0.8)
ax.axvline(0, color='0.8', linewidth=0.8)
ax.set_xlabel('Range residual / event range gate')
ax.set_ylabel('Range-rate residual / event rate gate')
ax.set_title('(c) Hybrid-normalized residual validation', loc='left', fontweight='bold')
ax = axes[1, 1]
x = np.arange(len(sensitivity))
bottom = np.zeros(len(sensitivity))
for status in ['unique', 'ambiguous', 'none']:
    values = sensitivity[status].to_numpy()
    bars = ax.bar(x, values, bottom=bottom, color=colors[status], label=status.capitalize())
    for bar, value, base in zip(bars, values, bottom):
        if value:
            ax.text(bar.get_x() + bar.get_width() / 2, base + value / 2, str(int(value)), ha='center', va='center', color='white' if status != 'none' else '#263238', fontsize=9, fontweight='bold')
    bottom += values
ax.set_xticks(x)
ax.set_xticklabels([label for label, _ in criteria])
ax.set_ylabel('Object detections')
ax.set_title('(d) Association-rule sensitivity', loc='left', fontweight='bold')
PANEL_PNG = OUT / 'catalog_association_diagnostics_2x2.png'
PANEL_PDF = OUT / 'catalog_association_diagnostics_2x2.pdf'
fig.savefig(PANEL_PNG, bbox_inches='tight')
fig.savefig(PANEL_PDF, bbox_inches='tight')
plt.show()
campaign_summary = {'detector': {'config_id': EXPECTED_CONFIG_ID, 'threshold': EXPECTED_THRESHOLD, 'track_rule': EXPECTED_TRACK_RULE}, 'time_scope': {'hourly_files': int(len(hourly_files)), 'start_utc': str(pd.to_datetime(objects.start_time_unix.min(), unit='s', utc=True)), 'end_utc': str(pd.to_datetime(objects.end_time_unix.max(), unit='s', utc=True))}, 'range_scope': {'gmf_min_range_gate': GMF_MIN_RANGE_GATE, 'gmf_max_range_gate': GMF_MAX_RANGE_GATE, 'gmf_range_gate_count': GMF_MAX_RANGE_GATE - GMF_MIN_RANGE_GATE, 'gmf_range_gate_source': GMF_RANGE_GATE_SOURCE, 'gmf_physical_range_km': [1019.444, 1091.245], 'cfar_valid_range_km': [1026.639, 1084.05], 'cfar_valid_width_km': 57.411}, 'object_detections': int(len(association_summary)), 'primary_hybrid_gate': {'unique_associations': int(len(matched)), 'ambiguous_associations': int(len(ambiguous)), 'unassociated_detections': int(len(unresolved)), 'unique_catalog_objects': int(unique_catalog.matched_NORAD_CAT_ID.nunique())}, 'published_fixed_gate': {'unique_associations': int(len(paper_matched)), 'ambiguous_associations': int(len(paper_ambiguous)), 'unassociated_detections': int(len(paper_unresolved))}, 'literature_benchmark': {'tromso_detections': 2077, 'catalog_correlated': 1411, 'uncorrelated': 666, 'figure_5_approximate_range_km': [250, 2000], 'comparison_warning': 'The present run covers all 24 temporal hours but only a 57.411 km CFAR-valid slant-range slice. It is not search-volume comparable.'}, 'association_rule': {'primary_range': '|dR| < max(1 km, 0.001*R_observed)', 'primary_range_rate': '|dRdot| < max(0.020 km/s, 0.01*abs(Rdot_observed))', 'beam': 'predicted beam separation <= 5 deg and above horizon', 'published_fixed_comparator': '|dR|<1 km and |dRdot|<20 m/s', 'equal_one_percent': 'diagnostic only; 1% range and floor-stabilized 1% range-rate'}}
(OUT / 'catalog_campaign_summary.json').write_text(json.dumps(campaign_summary, indent=2))
readme = f"24-hour CA-CFAR catalog association\n====================================\n\nFrozen detector: {EXPECTED_CONFIG_ID}, threshold {EXPECTED_THRESHOLD}, rule {EXPECTED_TRACK_RULE}\nObject detections cataloged: {len(association_summary)}\nUnique primary associations: {len(matched)}\nAmbiguous primary associations: {len(ambiguous)}\nUnassociated detections: {len(unresolved)}\nUnique matched NORAD objects: {unique_catalog.matched_NORAD_CAT_ID.nunique()}\n\nPrimary rule:\n- exact event time; no event-specific time fitting\n- latest TLE at or before each event; no future TLEs\n- |range residual| < max(1 km, 0.1% of observed slant range)\n- |range-rate residual| < max(20 m/s, 1% of absolute observed radial rate)\n- predicted line of sight within 5 degrees of the modeled beam/sidelobe region\n\nComparators:\n- the paper's fixed 1 km / 20 m/s gate is retained for direct comparison\n- equal 1% range and range-rate is diagnostic only because 1% of about\n  1050 km is approximately 10.5 km and raw velocity percentages are unstable\n  near zero radial rate\n\nScope warning:\nThe campaign contains all 24 temporal hours, but the completed GMF/CFAR search\nis restricted to approximately 1026.639-1084.050 km. Do not compare its raw\ncount directly with the literature's broader search volume or lower thresholds.\nGMF gates 6800-7280 were inherited from the Hardtarget example configuration,\nnot derived from the 2018 campaign.\n"
(OUT / 'README_catalog_association.txt').write_text(readme)
print('Primary hybrid percentage-based result')
print('  Object detections:', len(association_summary))
print('  Unique associations:', len(matched))
print('  Ambiguous:', len(ambiguous))
print('  Unassociated:', len(unresolved))
print('  Unique matched catalog objects:', unique_catalog.matched_NORAD_CAT_ID.nunique())
print('\nGate sensitivity:')
display(sensitivity)
print('\nSaved panel:', PANEL_PNG)


## 7. How to interpret the result
